# 📋 claude.ai 프롬프트 / claude.ai Prompt

> **수업 활용법:** 아래 프롬프트를 [claude.ai](https://claude.ai)에 붙여넣으면 유사한 노트북을 생성할 수 있습니다.

---

### 🇰🇷 한국어 프롬프트
```
Anthropic Claude API + feedparser(Google 뉴스 RSS)를 이용한
뉴스 검색 AI 에이전트 Google Colab 노트북을 한국어로 작성해주세요.

1. 개념: '외부 데이터 연동 에이전트' — 에이전트가 실시간 외부 데이터를
   가져와 LLM의 지식 한계를 극복하는 방법 설명
2. 도구: search_news(query, days=7)
   - feedparser로 Google 뉴스 RSS 검색
   - 최근 N일 이내 기사만 필터링
   - 제목·날짜·출처·링크 반환
3. 에이전트 루프 구현 (FM2-02와 동일한 패턴)
4. 에이전트 역할: 검색 결과에서 가장 중요한 뉴스 10개를 선정하고
   번호·제목·날짜·출처·링크를 표로 출력,
   10개 뉴스를 30단어 이내로 종합 요약
5. 테스트: 회사명(삼성전자, OpenAI 등)과 키워드(AI 반도체 등) 2가지
6. 미니 실습: 사용자가 직접 검색어를 입력(input())하여 뉴스 검색

설치: anthropic, feedparser. 한국어 주석. 초보자 친화적.
```

### 🇺🇸 English Prompt
```
Create a Korean Google Colab notebook for a News Search AI Agent
using Anthropic Claude API + feedparser (Google News RSS).

1. Concept: 'External Data Agent' — how agents overcome LLM knowledge cutoffs
2. Tool: search_news(query, days=7)
   - Fetch Google News RSS with feedparser
   - Filter articles from last N days
   - Return title, date, source, link
3. Implement agent loop (same pattern as FM2-02)
4. Agent behavior: from search results, select top 10 most important news,
   display as numbered list with title/date/source/link,
   summarize all 10 in ~30 words
5. Tests: a company name and a keyword query
6. Mini exercise: user types their own query via input()

Install: anthropic, feedparser. Korean comments. Beginner-friendly.
```

# FM2 실습 4: 뉴스 검색 AI 에이전트

## 학습 목표
- 에이전트가 실시간 외부 데이터를 가져오는 방법을 구현한다
- Google 뉴스 RSS를 도구로 연동하는 방법을 배운다
- Claude가 검색 결과를 분석·정리하는 과정을 이해한다

## 이 실습의 핵심 개념: 외부 데이터 연동 에이전트

| 한계 | 해결책 |
|------|--------|
| Claude의 학습 데이터는 특정 날짜에서 끊김 | 도구로 실시간 뉴스를 가져옴 |
| LLM은 최신 정보를 모름 | 에이전트가 인터넷에서 검색 후 Claude에 전달 |
| 환각(없는 정보 생성) 위험 | 실제 기사 URL을 출처로 제공 |

```
[에이전트 흐름]
"삼성전자 최근 뉴스 알려줘"
    ↓
Claude → search_news("삼성전자", days=7) 호출
    ↓
Google 뉴스 RSS → 기사 목록 반환
    ↓
Claude → 중요도 판단 → 상위 10개 선정 + 요약
```

In [ ]:
!pip install anthropic feedparser -q
import getpass, anthropic
import feedparser
import urllib.parse
from datetime import datetime, timezone, timedelta

api_key = getpass.getpass("🔑 Anthropic API 키: ")
client = anthropic.Anthropic(api_key=api_key)
print("✅ 준비 완료!")

## 1단계: 뉴스 검색 도구 정의

Google 뉴스는 **RSS 피드**를 무료로 제공합니다.  
`feedparser` 라이브러리로 RSS를 읽으면 별도 API 키 없이 뉴스를 가져올 수 있습니다.

In [ ]:
def search_news(query: str, days: int = 7) -> str:
    """
    Google 뉴스 RSS로 최근 기사를 검색합니다.

    Args:
        query: 검색할 키워드 또는 회사명
        days:  검색 기간 (최근 N일, 기본값 7)

    Returns:
        기사 목록 문자열 (제목·날짜·출처·링크)
    """
    encoded = urllib.parse.quote(query)
    url = f"https://news.google.com/rss/search?q={encoded}&hl=ko&gl=KR&ceid=KR:ko"

    feed = feedparser.parse(url)

    if not feed.entries:
        return f"'{query}'에 대한 뉴스를 찾을 수 없습니다."

    # 최근 N일 이내 기사만 필터링
    cutoff = datetime.now(timezone.utc) - timedelta(days=days)
    articles = []

    for entry in feed.entries:
        try:
            # feedparser는 published_parsed를 UTC time.struct_time으로 제공
            pub = datetime(*entry.published_parsed[:6], tzinfo=timezone.utc)
        except Exception:
            continue  # 날짜 파싱 실패 시 건너뜀

        if pub < cutoff:
            continue

        # 출처 이름 추출 (없으면 '출처 불명')
        source = getattr(getattr(entry, 'source', None), 'title', '출처 불명')

        articles.append({
            'title':  entry.title,
            'date':   pub.strftime('%Y-%m-%d'),
            'source': source,
            'link':   entry.link,
        })

    if not articles:
        return f"최근 {days}일 내 '{query}' 관련 기사가 없습니다."

    # 텍스트 형태로 반환 (Claude가 읽기 쉽게)
    lines = [f"🔍 '{query}' 검색 결과 — 최근 {days}일, 총 {len(articles)}건\n"]
    for i, a in enumerate(articles, 1):
        lines.append(f"{i}. [{a['date']}] {a['title']}")
        lines.append(f"   출처: {a['source']}")
        lines.append(f"   링크: {a['link']}")
        lines.append("")

    return "\n".join(lines)


# 도구 동작 확인
print("도구 테스트 중...")
test_result = search_news("인공지능", days=7)
print(test_result[:500], "...")

In [ ]:
# ─── 도구 스키마 + 매핑 ────────────────────────────────────────
TOOLS = [
    {
        "name": "search_news",
        "description": (
            "Google 뉴스에서 특정 키워드나 회사명으로 최근 뉴스를 검색합니다. "
            "최신 정보나 특정 회사/주제 관련 뉴스가 필요할 때 사용하세요."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "검색할 키워드 또는 회사명 (예: '삼성전자', 'AI 반도체')"
                },
                "days": {
                    "type": "integer",
                    "description": "검색 기간 (일 수, 기본값 7)"
                }
            },
            "required": ["query"]
        }
    }
]

TOOL_FUNCTIONS = {"search_news": search_news}

print("✅ 도구 스키마 준비 완료!")

## 2단계: 뉴스 에이전트 루프

In [ ]:
# 에이전트에게 줄 시스템 프롬프트
NEWS_SYSTEM_PROMPT = """당신은 뉴스 분석 전문 에이전트입니다.

역할:
1. search_news 도구로 관련 뉴스를 검색합니다
2. 검색 결과에서 가장 중요한 뉴스 최대 10개를 선정합니다
3. 다음 형식으로 출력합니다:

   ## 📰 [검색어] 주요 뉴스 TOP 10 (최근 7일)

   | # | 제목 | 날짜 | 출처 | 링크 |
   |---|------|------|------|------|
   | 1 | ... | ... | ... | [링크](URL) |
   ...

   ## 📝 종합 요약 (30단어 이내)
   ...

중요도 판단 기준: 업계 파급력, 사회적 관심도, 새로운 정보 여부"""


def run_news_agent(user_query, verbose=True):
    """뉴스 검색 에이전트를 실행합니다."""
    messages = [{"role": "user", "content": user_query}]

    if verbose:
        print(f"\n❓ 요청: {user_query}")
        print("─" * 65)

    while True:
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=2048,
            system=NEWS_SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages
        )

        messages.append({"role": "assistant", "content": response.content})

        # 최종 답변
        if response.stop_reason == "end_turn":
            final = next((b.text for b in response.content if hasattr(b, 'text')), "")
            print(final)
            return final

        # 도구 호출
        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue

                tool_input = block.input  # ✅ block.input 사용

                if verbose:
                    print(f"  🔧 뉴스 검색 중: {tool_input}")

                result = TOOL_FUNCTIONS[block.name](**tool_input)

                if verbose:
                    print(f"  📦 {result.split(chr(10))[0]}")

                tool_results.append({
                    "type":        "tool_result",
                    "tool_use_id": block.id,
                    "content":     result
                })

            messages.append({"role": "user", "content": tool_results})


print("✅ 뉴스 에이전트 준비 완료!")

## 3단계: 테스트

In [ ]:
# 테스트 1: 회사명으로 검색
run_news_agent("삼성전자 관련 최근 1주일 주요 뉴스 TOP 10을 보여줘")

In [ ]:
# 테스트 2: 키워드로 검색
run_news_agent("AI 반도체 최근 주요 뉴스 10개와 30단어 요약을 보여줘")

## ✏️ 미니 실습: 직접 검색해보기

궁금한 회사명이나 키워드를 입력하면 뉴스를 검색합니다.

In [ ]:
print("🔍 뉴스 검색 에이전트")
user_keyword = input("검색할 회사명 또는 키워드를 입력하세요: ").strip()

if user_keyword:
    run_news_agent(f"{user_keyword} 관련 최근 1주일 주요 뉴스 TOP 10과 종합 요약을 보여줘")
else:
    print("검색어를 입력해주세요.")

## 📝 정리

| 요소 | 이번 실습 | 역할 |
|------|----------|------|
| 도구 | `search_news()` | 실시간 외부 데이터 수집 |
| 데이터 소스 | Google 뉴스 RSS | 최신 뉴스 (무료, API 키 불필요) |
| 에이전트 역할 | 중요도 판단 + 요약 | LLM의 분석·정리 능력 활용 |

**에이전트가 해결하는 LLM의 한계:**
- ❌ LLM 단독: "삼성전자 최근 뉴스" → 학습 시점 이후 정보 없음
- ✅ 에이전트: search_news 도구 → 오늘 뉴스 → 요약 제공

**실제 적용 아이디어:**
- 경쟁사 모니터링 자동화
- 투자 관심 종목 뉴스 요약
- 업계 트렌드 주간 브리핑 자동 생성

**다음 실습 (FM2-05):** Wikipedia 리서치 에이전트 — 노트 저장 도구를 추가하여 더 정교한 다단계 조사를 구현합니다!